# S2 cointegration — leverage (vol-target policy)

This notebook chooses a **holdable** `target_ann_vol` on unlevered daily pre-VT book returns. Arithmetic Sharpe is roughly invariant to constant leverage; CAGR, Calmar, drawdown, and CVaR are not. It is **not** a prop-firm pass-rate study.

**Run first.** This notebook does not fetch, score, or rebuild the book: it only reads the exported unlevered base parquet. Missing file → loud error naming the export notebook and path.

- Frozen `04_backtest/s2_coint/artifacts/s2_star_stack.json` (the tearsheet already loads this). Star panels if cached; otherwise `01_star_tearsheet` rebuilds the OLS overlay from the census panel.
- Then run `04_backtest/s2_coint/notebooks/01_star_tearsheet.ipynb` so both exist:
  - `01_data/data_files/s2_coint/s2_period_returns_base.parquet` — pre-VT daily book `ret` (this notebook; full sample so IS/OOS can be split)
  - `01_data/data_files/s2_coint/s2_period_returns.parquet` — net post-VT (later MC / prop-firm)

Half-Kelly is betting about half the theoretically growth-optimal fraction so noisy estimates of edge do not blow the account; CAGR is the constant yearly rate that turns $1 into ending wealth; Calmar is that CAGR divided by the worst peak-to-trough loss; and CVaR is the average outcome in the worst tail (here 5%).

## 0. Imports & Config


In [1]:
import os
import sys

import pandas as pd
from IPython.display import display

cur = os.path.abspath(os.getcwd())
ROOT = cur
for _ in range(12):
    if os.path.isfile(os.path.join(cur, "pyproject.toml")) and os.path.isdir(
        os.path.join(cur, "06_risk")
    ):
        ROOT = cur
        break
    parent = os.path.dirname(cur)
    if parent == cur:
        break
    cur = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from risk.analytics.leverage.apply import s2_frozen_cfg
from risk.analytics.leverage.artifacts import artifact_path
from risk.analytics.leverage.loaders import load_s2_period_returns_base
from risk.analytics.leverage.plots import surface_cagr_figure, surface_calmar_figure, surface_dd_figure
from risk.analytics.leverage.report import run_leverage_policy
from risk.analytics.monte_carlo.loaders import find_repo_root

ROOT = find_repo_root(ROOT)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print("ROOT", ROOT)

SLEEVE = "s2"
BAR = "D"
PERIODS_PER_YEAR = 252
SIGMA_WINDOW = 60
VT_STAR = "s1_vt"
DEFAULT_IS_END = "2021-12-31"
DEFAULT_TARGETS = [0.06, 0.08, 0.10, 0.12, 0.15, 0.18]
DEFAULT_DD_CAP = 0.25
PICK = "calmar"
ARTIFACT_PATH = artifact_path(ROOT, SLEEVE)
CFG = s2_frozen_cfg(
    target_ann_vol=0.10,
    periods_per_year=PERIODS_PER_YEAR,
    sigma_window=SIGMA_WINDOW,
)
print("s2 vt family s1_vt sigma_window", SIGMA_WINDOW)
print("ARTIFACT_PATH", ARTIFACT_PATH)


ROOT c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio
s2 vt family s1_vt sigma_window 60
ARTIFACT_PATH c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\06_risk\analytics\leverage\artifacts\s2_leverage.json


## 1. Data Loading


In [2]:
BASE = load_s2_period_returns_base(ROOT)
print("base bars", len(BASE), BASE.index.min().date(), BASE.index.max().date())
print(BASE.tail())


base bars 1164 2022-01-03 2026-08-24
date
2026-08-18    0.005848
2026-08-19   -0.003568
2026-08-20    0.001238
2026-08-21   -0.001774
2026-08-24    0.000000
Name: base, dtype: float64


## 2. Vol-target surface

Frozen VT family; only `target_ann_vol` changes. IS and OOS are reported separately. Half-Kelly is a **ceiling**, not an objective.


In [3]:
PACK = {}

def run(is_end, max_oos_dd):
    pack = run_leverage_policy(
        BASE,
        CFG,
        targets=list(DEFAULT_TARGETS),
        is_end=is_end,
        periods_per_year=PERIODS_PER_YEAR,
        max_oos_dd=float(max_oos_dd),
        pick=PICK,
        artifact_path=ARTIFACT_PATH,
        sleeve=SLEEVE,
        vt_star=VT_STAR,
    )
    PACK.clear()
    PACK.update(pack)
    print("This overlay is live VT on **base** returns, not r'=k r on the sealed net parquet.")
    print("half-Kelly vol ceiling (not an objective)", pack["half_kelly_vol"])
    display(pack["surface"])
    display(pd.Series(pack["decision"], name="policy").to_frame("value"))
    display(surface_cagr_figure(pack["surface"]))
    display(surface_calmar_figure(pack["surface"]))
    display(surface_dd_figure(pack["surface"]))
    return pack

pack = run(DEFAULT_IS_END, DEFAULT_DD_CAP)
try:
    import ipywidgets as w
    ui = w.interactive(
        run,
        is_end=w.Text(value=str(DEFAULT_IS_END), description="IS end"),
        max_oos_dd=w.FloatSlider(min=0.05, max=0.50, value=DEFAULT_DD_CAP, step=0.05, description="DD veto"),
    )
    display(ui)
except Exception as exc:
    print("ipywidgets unavailable (%s); default run already executed" % exc)


This overlay is live VT on **base** returns, not r'=k r on the sealed net parquet.
half-Kelly vol ceiling (not an objective) 0.6786000285595805


,target_ann_vol,realized_vol,is_sharpe,is_cagr,is_max_drawdown,is_calmar,is_cvar,oos_sharpe,oos_cagr,oos_max_drawdown,oos_calmar,oos_cvar,oos_n,is_n
0,0.06,0.078954,NaN,NaN,NaN,NaN,NaN,1.486124,0.121052,-0.038461,3.147357,-0.007055,1164.0,0.0
1,0.08,0.100491,NaN,NaN,NaN,NaN,NaN,1.417704,0.147446,-0.047799,3.084722,-0.009024,1164.0,0.0
2,0.10,0.112622,NaN,NaN,NaN,NaN,NaN,1.405019,0.164207,-0.058174,2.822684,-0.010549,1164.0,0.0
3,0.12,0.120120,NaN,NaN,NaN,NaN,NaN,1.406280,0.175703,-0.071493,2.457607,-0.011631,1164.0,0.0
4,0.15,0.125342,NaN,NaN,NaN,NaN,NaN,1.396661,0.182187,-0.083690,2.176933,-0.012545,1164.0,0.0
5,0.18,0.126634,NaN,NaN,NaN,NaN,NaN,1.385268,0.182422,-0.083382,2.187800,-0.012844,1164.0,0.0


,value
target_ann_vol,0.06
pick,calmar
n_survivors,6
half_kelly_vol,0.6786
max_oos_dd,0.25
oos_calmar,3.147357
oos_cagr,0.121052
oos_max_drawdown,-0.038461
oos_sharpe,1.486124
reason,ok


interactive(children=(Text(value='2021-12-31', description='IS end'), FloatSlider(value=0.25, description='DD …

## 3. Policy pick


In [4]:
if not PACK:
    raise RuntimeError("run() did not populate PACK")
dec = PACK["decision"]
print("recommended target_ann_vol", dec.get("target_ann_vol"), "reason", dec.get("reason"))
print("survivors", dec.get("n_survivors"), "half-Kelly ceiling", PACK["half_kelly_vol"])
print("wrote", PACK["artifact_path"])
display(pd.Series(dec, name="policy").to_frame("value"))


recommended target_ann_vol 0.06 reason ok
survivors 6 half-Kelly ceiling 0.6786000285595805
wrote c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\06_risk\analytics\leverage\artifacts\s2_leverage.json


,value
target_ann_vol,0.06
pick,calmar
n_survivors,6
half_kelly_vol,0.6786
max_oos_dd,0.25
oos_calmar,3.147357
oos_cagr,0.121052
oos_max_drawdown,-0.038461
oos_sharpe,1.486124
reason,ok


## 4. Evaluation

- OOS Sharpe should be roughly flat across the vol grid (constant-$k$ invariance of arithmetic Sharpe).
- Pick is max OOS **Calmar** among targets that pass the drawdown veto and sit at or below half-Kelly vol.
- Label: live VT overlay on **base** returns (`s1_vt` family, only `target_ann_vol` changes).
- Next: `02_ev_vs_spy.ipynb` (geometry on sealed net) and `prop_firm/` (FTMO-capped $k$).


In [5]:
print("sleeve", SLEEVE, "pick", PICK)
print("half-Kelly is a ceiling, not an objective")
if PACK.get("decision"):
    print(PACK["decision"])


sleeve s2 pick calmar
half-Kelly is a ceiling, not an objective
{'target_ann_vol': 0.06, 'pick': 'calmar', 'n_survivors': 6, 'half_kelly_vol': 0.6786000285595805, 'max_oos_dd': 0.25, 'oos_calmar': 3.1473573309239673, 'oos_cagr': 0.12105190403531307, 'oos_max_drawdown': -0.03846144282567876, 'oos_sharpe': 1.486123847458473, 'reason': 'ok'}
